# Clustering Presupuestal Municipal (2022–2024) — Versión Optimizada

Este notebook realiza un análisis completo de clustering presupuestal de municipalidades peruanas con tres enfoques:

1. **Análisis Estático (K-Means)**: Agrupa municipios según indicadores presupuestales sin considerar tiempo
2. **Análisis de Panel**: Análisis longitudinal que considera la estructura temporal
3. **Análisis de Trayectorias**: Seguimiento de cambios en asignación de clusters

## Indicadores Analizados:
- **ind_eje**: Indicador de ejecución presupuestal
- **propim**: Proporción de inversión municipal
- **proinv**: Proporción de inversión total

**Autor**: Análisis automatizado  
**Fecha**: 2025  
**Dataset**: 1,770 municipalidades peruanas

In [ ]:
# @title 1. Instalación de Librerías Necesarias
print("Instalando librerías necesarias...")

# Instalar librerías si no están disponibles
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    print("✓ Todas las librerías están disponibles")
except ImportError as e:
    print(f"Instalando librería faltante: {e}")
    !pip install pandas numpy matplotlib seaborn scikit-learn -q
    print("✓ Instalación completada")

# Configuración de visualización
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n✓ Configuración completada")

In [ ]:
# @title 2. Carga del Archivo CSV 📁
import pandas as pd
import os

print("=" * 60)
print("CARGA DE DATOS - CLUSTERING PRESUPUESTAL MUNICIPAL")
print("=" * 60)

# Detectar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

# Opción 1: Si estamos en Google Colab, permitir subir archivo
if IN_COLAB:
    print("\n📌 Entorno detectado: Google Colab")
    print("\nOpciones de carga:")
    print("1. Subir archivo manualmente")
    print("2. Cargar desde Google Drive")
    print("3. Usar archivo del repositorio (si está clonado)")
    
    opcion = input("\nSeleccione opción (1/2/3): ").strip()
    
    if opcion == "1":
        from google.colab import files
        print("\n📤 Por favor, sube el archivo 'base.csv'")
        uploaded = files.upload()
        csv_filename = list(uploaded.keys())[0]
        print(f"✓ Archivo '{csv_filename}' cargado exitosamente")
        
    elif opcion == "2":
        from google.colab import drive
        drive.mount('/content/drive')
        print("\n📁 Google Drive montado")
        csv_path = input("Ingrese la ruta del archivo CSV en Drive (ej: /content/drive/MyDrive/base.csv): ")
        csv_filename = csv_path
        
    elif opcion == "3":
        csv_filename = "base.csv"
        if not os.path.exists(csv_filename):
            print(f"⚠️ Archivo '{csv_filename}' no encontrado en el directorio actual")
            print("Por favor, clone el repositorio o suba el archivo manualmente")
        else:
            print(f"✓ Usando archivo local: {csv_filename}")
else:
    # Opción 2: Si estamos en entorno local
    print("\n📌 Entorno detectado: Local/Jupyter")
    csv_filename = "base.csv"
    
    if os.path.exists(csv_filename):
        print(f"✓ Archivo '{csv_filename}' encontrado en el directorio actual")
    else:
        print(f"⚠️ Archivo '{csv_filename}' no encontrado")
        print("Asegúrese de que el archivo esté en el mismo directorio que este notebook")

# Cargar el archivo CSV
try:
    print(f"\n🔄 Cargando datos desde '{csv_filename}'...")
    df = pd.read_csv(csv_filename)
    print(f"✅ Datos cargados exitosamente!")
    print(f"\n📊 Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
    print(f"📋 Columnas: {', '.join(df.columns.tolist())}")
    
    # Mostrar información básica
    print("\n" + "=" * 60)
    print("INFORMACIÓN DEL DATASET")
    print("=" * 60)
    print(df.info())
    
    print("\n" + "=" * 60)
    print("PRIMERAS 5 FILAS")
    print("=" * 60)
    print(df.head())
    
except FileNotFoundError:
    print(f"❌ ERROR: No se pudo encontrar el archivo '{csv_filename}'")
    print("Por favor, verifique la ruta del archivo")
except Exception as e:
    print(f"❌ ERROR al cargar el archivo: {str(e)}")

In [ ]:
# @title 4. Visualizaciones Exploratorias 📈
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("Generando visualizaciones...")

# Crear figura con subplots
fig = plt.figure(figsize=(20, 12))

# 1. Distribución de indicadores brutos
indicadores = ['ind_eje', 'propim', 'proinv']
for i, ind in enumerate(indicadores, 1):
    plt.subplot(3, 4, i)
    plt.hist(df[ind], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title(f'Distribución de {ind}', fontsize=12, fontweight='bold')
    plt.xlabel(ind)
    plt.ylabel('Frecuencia')
    plt.grid(alpha=0.3)

# 2. Boxplots de indicadores
for i, ind in enumerate(indicadores, 1):
    plt.subplot(3, 4, i + 3)
    plt.boxplot(df[ind].dropna(), vert=True)
    plt.title(f'Boxplot: {ind}', fontsize=12, fontweight='bold')
    plt.ylabel('Valor')
    plt.grid(alpha=0.3)

# 3. Scatter plots entre indicadores
plt.subplot(3, 4, 7)
plt.scatter(df['ind_eje'], df['propim'], alpha=0.5, s=10)
plt.xlabel('ind_eje')
plt.ylabel('propim')
plt.title('ind_eje vs propim', fontsize=12, fontweight='bold')
plt.grid(alpha=0.3)

plt.subplot(3, 4, 8)
plt.scatter(df['ind_eje'], df['proinv'], alpha=0.5, s=10)
plt.xlabel('ind_eje')
plt.ylabel('proinv')
plt.title('ind_eje vs proinv', fontsize=12, fontweight='bold')
plt.grid(alpha=0.3)

plt.subplot(3, 4, 9)
plt.scatter(df['propim'], df['proinv'], alpha=0.5, s=10)
plt.xlabel('propim')
plt.ylabel('proinv')
plt.title('propim vs proinv', fontsize=12, fontweight='bold')
plt.grid(alpha=0.3)

# 4. Matriz de correlación
plt.subplot(3, 4, 10)
corr_matrix = df[indicadores].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            square=True, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlación', fontsize=12, fontweight='bold')

# 5. Distribución de clusters existentes
plt.subplot(3, 4, 11)
df['km_static'].value_counts().sort_index().plot(kind='bar', color='steelblue', alpha=0.7)
plt.title('Distribución Cluster Estático', fontsize=12, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Frecuencia')
plt.xticks(rotation=0)
plt.grid(alpha=0.3)

plt.subplot(3, 4, 12)
df['q'].value_counts().sort_index().plot(kind='bar', color='coral', alpha=0.7)
plt.title('Distribución Cluster Panel', fontsize=12, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Frecuencia')
plt.xticks(rotation=0)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('visualizaciones_exploratorias.png', dpi=300, bbox_inches='tight')
print("✓ Visualizaciones guardadas en 'visualizaciones_exploratorias.png'")
plt.show()

In [ ]:
# @title 6. Visualización de Clusters con PCA 🎨
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

print("Generando visualización de clusters con PCA...")

# Aplicar PCA para reducir a 2 dimensiones
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Crear visualización
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Gráfico 1: Clusters con PCA
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], 
                          c=clusters, cmap='viridis', 
                          alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} varianza)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} varianza)', fontsize=12)
axes[0].set_title('Clusters K-Means (Proyección PCA)', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Agregar centroides
centroides_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroides_pca[:, 0], centroides_pca[:, 1], 
                c='red', marker='X', s=300, edgecolors='black', 
                linewidth=2, label='Centroides')
axes[0].legend(fontsize=10)

# Agregar colorbar
cbar = plt.colorbar(scatter, ax=axes[0])
cbar.set_label('Cluster', fontsize=11)

# Gráfico 2: Varianza explicada por componentes principales
explained_var = pca.explained_variance_ratio_
axes[1].bar(['PC1', 'PC2'], explained_var, color=['steelblue', 'coral'], alpha=0.7)
axes[1].set_ylabel('Varianza Explicada', fontsize=12)
axes[1].set_title('Varianza Explicada por Componentes Principales', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

# Agregar texto con varianza total
total_var = sum(explained_var)
axes[1].text(0.5, max(explained_var) * 0.9, 
             f'Varianza Total Explicada: {total_var:.2%}',
             ha='center', fontsize=11, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('clusters_pca_visualization.png', dpi=300, bbox_inches='tight')
print("✓ Visualización guardada en 'clusters_pca_visualization.png'")
plt.show()

# Matriz de componentes principales
print("\n📊 CONTRIBUCIÓN DE VARIABLES A COMPONENTES PRINCIPALES:")
components_df = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=indicadores
)
print(components_df)
print(f"\n✓ Varianza total explicada por PC1 y PC2: {total_var:.2%}")

In [ ]:
# @title 8. Análisis de Clustering de Trayectorias 🛤️
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 60)
print("ANÁLISIS DE CLUSTERING DE TRAYECTORIAS")
print("=" * 60)

# Análisis de clusters de trayectorias (columna 'qa')
print("\n📊 Análisis de clusters de trayectorias existentes (columna 'qa'):")
print(f"\nDistribución de clusters de trayectorias:")
traj_dist = df['qa'].value_counts().sort_index()
print(traj_dist)

# Análisis por cluster de trayectorias
print("\n📊 CARACTERÍSTICAS PROMEDIO POR CLUSTER DE TRAYECTORIAS:")
for cluster_id in sorted(df['qa'].unique()):
    print(f"\n--- CLUSTER TRAYECTORIAS {cluster_id} ---")
    cluster_data = df[df['qa'] == cluster_id][['ind_eje', 'propim', 'proinv']]
    print(cluster_data.describe())

# Análisis de transiciones entre clusters
print("\n🔄 ANÁLISIS DE TRANSICIONES ENTRE MÉTODOS:")

# Comparación triple
print("\n1. Clustering Estático vs Trayectorias:")
comp_static_traj = pd.crosstab(df['km_static'], df['qa'], 
                                rownames=['Estático'], 
                                colnames=['Trayectorias'])
print(comp_static_traj)

print("\n2. Clustering Panel vs Trayectorias:")
comp_panel_traj = pd.crosstab(df['q'], df['qa'], 
                               rownames=['Panel'], 
                               colnames=['Trayectorias'])
print(comp_panel_traj)

# Visualizaciones
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Distribución de clusters de trayectorias
traj_dist.plot(kind='bar', ax=axes[0, 0], color='purple', alpha=0.7)
axes[0, 0].set_title('Distribución de Clusters de Trayectorias', 
                     fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Cluster de Trayectorias', fontsize=12)
axes[0, 0].set_ylabel('Frecuencia', fontsize=12)
axes[0, 0].grid(alpha=0.3, axis='y')
axes[0, 0].tick_params(axis='x', rotation=0)

# 2. Comparación Estático vs Trayectorias
sns.heatmap(comp_static_traj, annot=True, fmt='d', cmap='Blues', 
            ax=axes[0, 1], cbar_kws={'label': 'Frecuencia'})
axes[0, 1].set_title('Estático vs Trayectorias', fontsize=14, fontweight='bold')

# 3. Comparación Panel vs Trayectorias
sns.heatmap(comp_panel_traj, annot=True, fmt='d', cmap='Greens', 
            ax=axes[1, 0], cbar_kws={'label': 'Frecuencia'})
axes[1, 0].set_title('Panel vs Trayectorias', fontsize=14, fontweight='bold')

# 4. Comparación de las tres distribuciones
dist_comparison = pd.DataFrame({
    'Estático': df['km_static'].value_counts().sort_index(),
    'Panel': df['q'].value_counts().sort_index(),
    'Trayectorias': df['qa'].value_counts().sort_index()
})
dist_comparison.plot(kind='bar', ax=axes[1, 1], alpha=0.7)
axes[1, 1].set_title('Comparación de Distribuciones de Clusters', 
                     fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Cluster ID', fontsize=12)
axes[1, 1].set_ylabel('Frecuencia', fontsize=12)
axes[1, 1].legend(title='Método', fontsize=10)
axes[1, 1].grid(alpha=0.3, axis='y')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('analisis_trayectorias.png', dpi=300, bbox_inches='tight')
print("\n✓ Visualización guardada en 'analisis_trayectorias.png'")
plt.show()

print("\n" + "=" * 60)
print("✓ Análisis de clustering de trayectorias completado")
print("=" * 60)

In [ ]:
# @title 10. Comprimir y Descargar Todos los Resultados 📥
import shutil
import os

print("=" * 60)
print("COMPRESIÓN Y DESCARGA DE RESULTADOS")
print("=" * 60)

# Verificar que el directorio de resultados existe
if not os.path.exists('results'):
    print("⚠️ El directorio 'results' no existe. Ejecute primero la celda anterior.")
else:
    # Crear archivo ZIP con todos los resultados
    print("\n🗜️ Comprimiendo archivos...")
    
    # Listar archivos PNG generados (visualizaciones)
    png_files = [f for f in os.listdir('.') if f.endswith('.png')]
    
    # Copiar archivos PNG al directorio results
    for png_file in png_files:
        shutil.copy(png_file, os.path.join('results', png_file))
        print(f"✓ Copiado: {png_file}")
    
    # Crear ZIP
    zip_path = shutil.make_archive('resultados_clustering_municipal', 'zip', 'results')
    print(f"\n✅ Archivo ZIP generado: {zip_path}")
    
    # Mostrar contenido del ZIP
    print(f"\n📦 Contenido del archivo ZIP:")
    for file in sorted(os.listdir('results')):
        file_path = os.path.join('results', file)
        size_kb = os.path.getsize(file_path) / 1024
        print(f"   - {file} ({size_kb:.2f} KB)")
    
    # Tamaño total del ZIP
    zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"\n📊 Tamaño total del ZIP: {zip_size_mb:.2f} MB")
    
    # Intentar descargar en Google Colab
    print("\n📥 Intentando descargar archivo...")
    try:
        from google.colab import files
        files.download(zip_path)
        print("✓ Descarga iniciada en Google Colab")
    except Exception as e:
        print("ℹ️ No estamos en Google Colab o la descarga automática falló")
        print(f"📁 El archivo ZIP está disponible en: {os.path.abspath(zip_path)}")
        print("   Puede descargarlo manualmente desde el explorador de archivos")

print("\n" + "=" * 60)
print("✓ ANÁLISIS COMPLETADO")
print("=" * 60)
print("\n🎉 ¡Análisis de clustering presupuestal municipal finalizado con éxito!")
print("\n📋 Resumen de lo realizado:")
print("   ✓ Carga y exploración de datos")
print("   ✓ Análisis descriptivo y visualizaciones exploratorias")
print("   ✓ Clustering estático con K-Means")
print("   ✓ Visualización con PCA")
print("   ✓ Análisis de clustering de panel")
print("   ✓ Análisis de clustering de trayectorias")
print("   ✓ Exportación de resultados en CSV")
print("   ✓ Generación de visualizaciones (PNG)")
print("   ✓ Compresión en archivo ZIP")
print("\n📊 Dataset: 1,770 municipalidades peruanas")
print("📈 Métodos: 3 enfoques de clustering")
print("📁 Resultados: Disponibles en resultados_clustering_municipal.zip")

In [ ]:
# @title 9. Resumen Final y Exportación de Resultados 📦
import pandas as pd
import os

print("=" * 60)
print("RESUMEN FINAL DEL ANÁLISIS")
print("=" * 60)

# Crear DataFrame con todos los resultados
df_resultados = df.copy()

# Agregar el nuevo clustering estático si existe
if 'cluster_estatico_nuevo' in df_clustered.columns:
    # Fusionar con df_resultados usando el índice
    df_resultados = df_resultados.join(df_clustered['cluster_estatico_nuevo'], how='left')

print("\n📊 RESUMEN DE CLUSTERS:")
print(f"\n1. Total de municipalidades analizadas: {len(df)}")
print(f"\n2. Distribución por método:")
print(f"   - Clustering Estático (km_static): {df['km_static'].nunique()} clusters")
print(f"   - Clustering Panel (q): {df['q'].nunique()} clusters")
print(f"   - Clustering Trayectorias (qa): {df['qa'].nunique()} clusters")

# Estadísticas por indicador
print(f"\n3. Estadísticas Generales de Indicadores:")
print("\n   Indicador de Ejecución (ind_eje):")
print(f"   - Media: {df['ind_eje'].mean():.4f}")
print(f"   - Mediana: {df['ind_eje'].median():.4f}")
print(f"   - Desv. Estándar: {df['ind_eje'].std():.4f}")

print("\n   Proporción Inversión Municipal (propim):")
print(f"   - Media: {df['propim'].mean():.4f}")
print(f"   - Mediana: {df['propim'].median():.4f}")
print(f"   - Desv. Estándar: {df['propim'].std():.4f}")

print("\n   Proporción Inversión (proinv):")
print(f"   - Media: {df['proinv'].mean():.4f}")
print(f"   - Mediana: {df['proinv'].median():.4f}")
print(f"   - Desv. Estándar: {df['proinv'].std():.4f}")

# Crear directorio para resultados
os.makedirs('results', exist_ok=True)

# Exportar DataFrames
print("\n💾 Exportando resultados...")

# 1. Dataset completo con todos los clusters
df_resultados.to_csv('results/dataset_completo_con_clusters.csv', index=False)
print("✓ Dataset completo exportado")

# 2. Resumen por cluster estático
resumen_estatico = df.groupby('km_static')[['ind_eje', 'propim', 'proinv']].agg(['mean', 'median', 'std', 'count'])
resumen_estatico.to_csv('results/resumen_cluster_estatico.csv')
print("✓ Resumen cluster estático exportado")

# 3. Resumen por cluster panel
resumen_panel = df.groupby('q')[['ind_eje', 'propim', 'proinv']].agg(['mean', 'median', 'std', 'count'])
resumen_panel.to_csv('results/resumen_cluster_panel.csv')
print("✓ Resumen cluster panel exportado")

# 4. Resumen por cluster trayectorias
resumen_traj = df.groupby('qa')[['ind_eje', 'propim', 'proinv']].agg(['mean', 'median', 'std', 'count'])
resumen_traj.to_csv('results/resumen_cluster_trayectorias.csv')
print("✓ Resumen cluster trayectorias exportado")

# 5. Matriz de comparación entre métodos
comparacion = pd.crosstab([df['km_static'], df['q']], df['qa'])
comparacion.to_csv('results/comparacion_metodos.csv')
print("✓ Comparación entre métodos exportada")

print("\n📁 Archivos generados en directorio 'results/':")
for file in sorted(os.listdir('results')):
    file_path = os.path.join('results', file)
    size_kb = os.path.getsize(file_path) / 1024
    print(f"   - {file} ({size_kb:.2f} KB)")

print("\n" + "=" * 60)
print("✓ Exportación de resultados completada")
print("=" * 60)

In [ ]:
# @title 7. Análisis de Clustering de Panel 📋
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("ANÁLISIS DE CLUSTERING DE PANEL")
print("=" * 60)

# Nota: El clustering de panel requiere datos longitudinales (múltiples periodos)
# En este caso, trabajaremos con los clusters de panel existentes en los datos (columna 'q')

print("\n📊 Análisis de clusters de panel existentes (columna 'q'):")
print(f"\nDistribución de clusters de panel:")
panel_dist = df['q'].value_counts().sort_index()
print(panel_dist)

# Análisis por cluster de panel
print("\n📊 CARACTERÍSTICAS PROMEDIO POR CLUSTER DE PANEL:")
for cluster_id in sorted(df['q'].unique()):
    print(f"\n--- CLUSTER PANEL {cluster_id} ---")
    cluster_data = df[df['q'] == cluster_id][['ind_eje', 'propim', 'proinv']]
    print(cluster_data.describe())

# Comparación entre clustering estático y de panel
print("\n🔍 COMPARACIÓN: Clustering Estático vs Panel")
comparison = pd.crosstab(df['km_static'], df['q'], 
                         rownames=['Cluster Estático'], 
                         colnames=['Cluster Panel'])
print("\nTabla de contingencia:")
print(comparison)

# Visualización de la comparación
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Distribución de clusters de panel
panel_dist.plot(kind='bar', ax=axes[0], color='teal', alpha=0.7)
axes[0].set_title('Distribución de Clusters de Panel', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Cluster de Panel', fontsize=12)
axes[0].set_ylabel('Frecuencia', fontsize=12)
axes[0].grid(alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=0)

# Gráfico 2: Heatmap de comparación
import seaborn as sns
sns.heatmap(comparison, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1], 
            cbar_kws={'label': 'Frecuencia'})
axes[1].set_title('Clustering Estático vs Panel', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Cluster Panel', fontsize=12)
axes[1].set_ylabel('Cluster Estático', fontsize=12)

plt.tight_layout()
plt.savefig('analisis_panel.png', dpi=300, bbox_inches='tight')
print("\n✓ Visualización guardada en 'analisis_panel.png'")
plt.show()

print("\n" + "=" * 60)
print("✓ Análisis de clustering de panel completado")
print("=" * 60)

In [ ]:
# @title 5. Clustering Estático con K-Means 🎯
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

print("=" * 60)
print("ANÁLISIS DE CLUSTERING ESTÁTICO (K-MEANS)")
print("=" * 60)

# Preparar datos para clustering
indicadores = ['ind_eje', 'propim', 'proinv']
X = df[indicadores].copy()

# Eliminar filas con valores faltantes
X_clean = X.dropna()
print(f"\n📊 Datos para clustering: {X_clean.shape[0]} municipalidades")

# Normalizar datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

# Método del codo para determinar número óptimo de clusters
print("\n🔍 Calculando método del codo...")
inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Visualizar método del codo
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (k)', fontsize=12)
plt.ylabel('Inercia (Within-Cluster Sum of Squares)', fontsize=12)
plt.title('Método del Codo para Determinar K Óptimo', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.savefig('metodo_codo.png', dpi=300, bbox_inches='tight')
print("✓ Gráfico del método del codo guardado")
plt.show()

# Aplicar K-Means con k óptimo (usaremos k=3 como en los datos originales)
k_optimo = 3
print(f"\n🎯 Aplicando K-Means con k={k_optimo}...")

kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Agregar clusters al DataFrame
df_clustered = X_clean.copy()
df_clustered['cluster_estatico_nuevo'] = clusters

# Análisis de clusters
print(f"\n📈 DISTRIBUCIÓN DE CLUSTERS:")
print(df_clustered['cluster_estatico_nuevo'].value_counts().sort_index())

print(f"\n📊 CENTROIDES DE CLUSTERS (valores normalizados):")
centroides = kmeans.cluster_centers_
centroides_df = pd.DataFrame(centroides, columns=indicadores)
centroides_df.index = [f'Cluster {i}' for i in range(k_optimo)]
print(centroides_df)

# Desnormalizar centroides para interpretación
print(f"\n📊 CENTROIDES DE CLUSTERS (valores originales):")
centroides_orig = scaler.inverse_transform(centroides)
centroides_orig_df = pd.DataFrame(centroides_orig, columns=indicadores)
centroides_orig_df.index = [f'Cluster {i}' for i in range(k_optimo)]
print(centroides_orig_df)

# Características promedio por cluster
print(f"\n📊 ESTADÍSTICAS POR CLUSTER:")
for i in range(k_optimo):
    print(f"\n--- CLUSTER {i} ---")
    cluster_data = df_clustered[df_clustered['cluster_estatico_nuevo'] == i][indicadores]
    print(cluster_data.describe())

print("\n" + "=" * 60)
print("✓ Análisis de clustering estático completado")
print("=" * 60)

In [ ]:
# @title 3. Exploración y Análisis Descriptivo de Datos 📊
import pandas as pd
import numpy as np

print("=" * 60)
print("ANÁLISIS EXPLORATORIO DE DATOS")
print("=" * 60)

# Verificar valores faltantes
print("\n1️⃣ VALORES FALTANTES:")
print(df.isnull().sum())

# Estadísticas descriptivas de los indicadores principales
print("\n2️⃣ ESTADÍSTICAS DESCRIPTIVAS - INDICADORES BRUTOS:")
indicadores_brutos = ['ind_eje', 'propim', 'proinv']
print(df[indicadores_brutos].describe())

print("\n3️⃣ ESTADÍSTICAS DESCRIPTIVAS - INDICADORES NORMALIZADOS (Z-SCORES):")
indicadores_normalizados = ['z_ind_eje', 'z_propim', 'z_proinv']
print(df[indicadores_normalizados].describe())

# Distribución de clusters existentes
print("\n4️⃣ DISTRIBUCIÓN DE CLUSTERS EXISTENTES:")
print("\nCluster Estático (km_static):")
print(df['km_static'].value_counts().sort_index())

print("\nCluster de Panel (q):")
print(df['q'].value_counts().sort_index())

print("\nCluster de Trayectorias (qa):")
print(df['qa'].value_counts().sort_index())

# Correlación entre indicadores
print("\n5️⃣ MATRIZ DE CORRELACIÓN:")
correlacion = df[indicadores_brutos].corr()
print(correlacion)

print("\n" + "=" * 60)
print("✓ Análisis exploratorio completado")
print("=" * 60)